In [ ]:
import os, sys
from pathlib import Path
sys.path.append('../src')
sys.path.append('../src/utils/')
from lava_network.spiking_dataloader import WISDM_spiking_dataloader, WisdmDatasetParser
from lava_network.output_process import OutputProcess
import numpy as np


In [ ]:


#path = f"{Path.home()}/snntorch_network/notebook/Trained/network_best.npz"
path = f"{Path.home()}/snntorch_network/nni_experiments/Inibitory_lif_no_encoder/results/hjulef4s/trials/RHEsE/Trained/network_best.npz"
data = np.load(path,allow_pickle=True)


linear1_w= data['linear1']
leaky1_vth= data['leaky1_vth']
leaky1_betas= 1-data['leaky1_betas'] 
leaky1_betas= leaky1_betas if leaky1_betas >= 0 else np.zeros(leaky1_betas.shape)
print(f"leaky1_betas: {leaky1_betas}")
print(f"leaky1_vth: {leaky1_vth}")
linear2_w = data['linear2']
leaky2_vth= data['recurrent_vth']
leaky2_betas= 1 - data['recurrent_betas']
leaky2_betas= leaky2_betas if  leaky2_betas >= 0 else np.zeros(leaky2_betas.shape)
print(f"leaky2_betas: {leaky2_betas}")
print(f"leaky2_vth: {leaky2_vth}")

recurrent_in_weights = data['input_dense']
recurrent_out_weights = - data['output_dense']
recurrent_vth = data['activation_vth']
recurrent_leaky_betas = 1 - data['activation_betas']
recurrent_leaky_betas= recurrent_leaky_betas if recurrent_leaky_betas >= 0 else np.zeros(recurrent_leaky_betas.shape)
print(f"recurrent_leaky_betas: {recurrent_leaky_betas}")
print(f"recurrent_vth: {recurrent_vth}")

linear3_w = data['linear3']
leaky3_vth= data['leaky2_vth']
leaky3_betas= 1 - data['leaky2_betas']
leaky3_betas= leaky3_betas if leaky3_betas >= 0 else np.zeros(leaky3_betas.shape)
print(f"leaky3_betas: {leaky3_betas}")
print(f"leaky3_vth: {leaky3_vth}")


leaky1_betas: 0.8172026127576828
leaky1_vth: 1.711238145828247
leaky2_betas: 0.1649174690246582
leaky2_vth: 0.5812320113182068
recurrent_leaky_betas: 0.13742202520370483
recurrent_vth: 1.1815392971038818
leaky3_betas: 0.0
leaky3_vth: 1.5450438261032104


In [4]:
import json
def deserialize_dict(json_str):
    def convert(obj):
        if isinstance(obj, list):
            return np.array(obj).astype(np.int64)
        if isinstance(obj, int):
            return np.int64(obj)
        return obj
    
    return json.loads(json_str, object_hook=lambda d: {k: convert(v) for k, v in d.items()})


In [5]:
with open('network_fixed.json', 'r') as json_file:
    loaded_json_str = json_file.read()
converted_params = deserialize_dict(loaded_json_str)

In [6]:
from lava.proc.lif.process import LIF
from lava.proc.dense.process import Dense, LearningDense 
from lava.utils.weightutils import SignMode   
from lif_mod import LIFEncoder
from dense_mod import DenseEncoder

linear1 = DenseEncoder(weights=linear1_w, num_message_bits=32, name="linear1")

leaky1 = LIFEncoder(shape=(linear1_w.shape[0],),
                    u = np.zeros(linear1_w.shape[0]),
                    v = np.zeros(linear1_w.shape[0]),
                    du = 1.0,
                    dv = leaky1_betas,
                    vth=leaky1_vth,
                    log_config=0,
                    name= "leaky1"
                )
linear1.a_out.connect(leaky1.a_in)
name = "linear2"
linear2 = Dense(**converted_params[name],
                sign_mode=SignMode.MIXED, name=name)

linear2.s_in.connect_from(leaky1.s_out)

name = "leaky2"
leaky2 = LIF(shape=(linear2_w.shape[0],),
                    **converted_params[name],
                    name= name
                )
#sum.a_out.connect(leaky2.a_in)
linear2.a_out.connect(leaky2.a_in)
#leaky2.a_in.connect_from(linear2.a_out)
name = "recurrent_in"
recurrent_in = Dense(**converted_params[name],
                     name=name)

leaky2.s_out.connect(recurrent_in.s_in)

name = "inibitory_leaky"
ahpc = LIF(shape=(recurrent_in_weights.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )

recurrent_in.a_out.connect(ahpc.a_in)
name = "recurrent_out"
recurrent_out = Dense(**converted_params[name],
                       name=name)
recurrent_out.s_in.connect_from(ahpc.s_out)
recurrent_out.a_out.connect(leaky2.a_in)
name = "linear3"
linear3 = Dense(**converted_params[name],
                        name=name)

linear3.s_in.connect_from(leaky2.s_out)
name = "leaky3"
leaky3 = LIF(shape=(linear3_w.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )
leaky3.a_in.connect_from(linear3.a_out)


In [7]:
num_samples = 3000
signal_step = 40
clear_intervall = 4
train_percentage = 0.6
time_steps = signal_step + clear_intervall

dataset = WisdmDatasetParser('../data/data_watch_subset_0_40.npz', norm=None, class_sublset='custom', subset_list=[0, 4, 6, 8, 9, 10, 14])
val_set = dataset.get_validation_set(shuffle=False, subset=num_samples)
#val_set = dataset.get_validation_set()
val_set[0].shape
num_samples = val_set[0].shape[0]
spiking_loader = WISDM_spiking_dataloader(val_set ,clear_intervall=clear_intervall)
out_sink = OutputProcess(7,num_samples,time_steps, 0)

(6,)
(6,)
ytrain shape (55404, 18)
yval shape (18468, 18)
ytest shape (18469, 18)
num classes train dataset: 7 occurrences of each class:[3127 3044 3102 3047 3150 3087 2973]
num classes eval dataset: 7 occurrences of each class:[1035 1048 1122  996 1110 1053 1007]
num classes test dataset: 7 occurrences of each class:[1046 1048 1046 1036 1076 1026  982]


In [8]:
spiking_loader.data_out.connect(linear1.s_in)
leaky3.s_out.connect(out_sink.spikes_in)
out_sink.label_in.connect_from(spiking_loader.label_out)

In [9]:

from lava.magma.core.run_conditions import RunSteps
from lava.magma.core.run_configs import Loihi1SimCfg, Loihi2SimCfg
import numpy as np
from tqdm import tqdm

for i in tqdm(range(num_samples)):
      out_sink.run(condition=RunSteps(num_steps=time_steps),
                  run_cfg=Loihi1SimCfg(select_sub_proc_model=True,
                  select_tag='fixed_pt'))
      leaky1.v.set(np.zeros(leaky1.v.shape))
      leaky1.u.set(np.zeros(leaky1.u.shape))
      leaky2.v.set(np.zeros(leaky2.v.shape))
      leaky2.u.set(np.zeros(leaky2.u.shape))
      leaky3.v.set(np.zeros(leaky3.v.shape))
      leaky3.u.set(np.zeros(leaky3.u.shape))
      ahpc.v.set(np.zeros(ahpc.v.shape))
      ahpc.u.set(np.zeros(ahpc.u.shape))
      linear1.a_buff.set(np.zeros(linear1.a_buff.shape))
      linear2.a_buff.set(np.zeros(linear2.a_buff.shape))
      linear3.a_buff.set(np.zeros(linear3.a_buff.shape))
        
ground_truth = val_set[1][:num_samples]
pre_predictions = out_sink.pred_labels.get().astype(int)
# Stop the execution
out_sink.stop()

accuracy = np.sum(ground_truth==pre_predictions)/ground_truth.size * 100

print(f"\nGround truth: {ground_truth}\n"
      f"Predictions : {pre_predictions}\n"
      f"Accuracy    : {accuracy}")

100%|██████████| 3000/3000 [03:12<00:00, 15.62it/s]



Ground truth: [2 5 4 ... 2 0 1]
Predictions : [2 5 3 ... 2 0 6]
Accuracy    : 93.2


In [10]:
# Calculate statistics for linear3.weights.init
weights = linear3.weights.init

mean_value = np.mean(weights)
min_value = np.min(weights)
max_value = np.max(weights)
std_dev = np.std(weights)

print(f"Mean value: {mean_value}")
print(f"Minimum value: {min_value}")
print(f"Maximum value: {max_value}")
print(f"Standard Deviation: {std_dev}")

Mean value: -1.5635714285714286
Minimum value: -81
Maximum value: 63
Standard Deviation: 11.483445915840802


In [11]:
from lava.proc.learning_rules.r_stdp_learning_rule import RewardModulatedSTDP
from lava.proc.learning_rules.stdp_learning_rule import STDPLoihi as STDP

s_stdp = STDP(learning_rate=7,
            A_plus=8,
            A_minus=-4,
            tau_plus=5,
            tau_minus=3,
            t_epoch=int(float(num_samples)*train_percentage)*time_steps,
            x1_impulse= 0,
            y1_impulse= 0,
            )

R_STDP = RewardModulatedSTDP(learning_rate=2,
                             A_plus=4,
                             A_minus=-4,
                             pre_trace_decay_tau=8,
                             post_trace_decay_tau=8, 
                             pre_trace_kernel_magnitude=10,
                             post_trace_kernel_magnitude=10,
                             eligibility_trace_decay_tau=0.5,
                             t_epoch=100*128
                             )

In [12]:
linear1 = DenseEncoder(weights=linear1_w, num_message_bits=32, name="linear1")

leaky1 = LIFEncoder(shape=(linear1_w.shape[0],),
                    u = np.zeros(linear1_w.shape[0]),
                    v = np.zeros(linear1_w.shape[0]),
                    du = 1.0,
                    dv = leaky1_betas,
                    vth=leaky1_vth,
                    log_config=0,
                    name= "leaky1"
                )
linear1.a_out.connect(leaky1.a_in)
name = "linear2"
linear2 = Dense(**converted_params[name],
                sign_mode=SignMode.MIXED, name=name)

linear2.s_in.connect_from(leaky1.s_out)

name = "leaky2"
leaky2 = LIF(shape=(linear2_w.shape[0],),
                    **converted_params[name],
                    name= name
                )
#sum.a_out.connect(leaky2.a_in)
linear2.a_out.connect(leaky2.a_in)
#leaky2.a_in.connect_from(linear2.a_out)
name = "recurrent_in"
recurrent_in = Dense(**converted_params[name],
                     name=name)

leaky2.s_out.connect(recurrent_in.s_in)

name = "inibitory_leaky"
ahpc = LIF(shape=(recurrent_in_weights.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )

recurrent_in.a_out.connect(ahpc.a_in)

name = "recurrent_out"
recurrent_out = Dense(**converted_params[name],
                       name=name)
recurrent_out.s_in.connect_from(ahpc.s_out)
recurrent_out.a_out.connect(leaky2.a_in)

name = "linear3"
linear3 = LearningDense(**converted_params[name],
                        learning_rule=s_stdp,
                        name=name)

linear3.s_in.connect_from(leaky2.s_out)
name = "leaky3"
leaky3 = LIF(shape=(linear3_w.shape[0],),
                    **converted_params[name],
                    log_config=0,
                    name= name
                )
leaky3.s_out.connect(linear3.s_in_bap)
leaky3.a_in.connect_from(linear3.a_out)



In [13]:
dataset = WisdmDatasetParser('../data/data_watch_subset_0_40.npz', norm=None, class_sublset='custom', subset_list=[0, 4, 6, 8, 9, 10, 14])
val_set = dataset.get_validation_set(shuffle=False, subset=num_samples)
#val_set = dataset.get_validation_set()
val_set[0].shape
num_samples = val_set[0].shape[0]
spiking_loader = WISDM_spiking_dataloader(val_set ,clear_intervall=clear_intervall)
out_sink = OutputProcess(7,num_samples,time_steps, 0)

(6,)
(6,)
ytrain shape (55404, 18)
yval shape (18468, 18)
ytest shape (18469, 18)
num classes train dataset: 7 occurrences of each class:[3127 3044 3102 3047 3150 3087 2973]
num classes eval dataset: 7 occurrences of each class:[1035 1048 1122  996 1110 1053 1007]
num classes test dataset: 7 occurrences of each class:[1046 1048 1046 1036 1076 1026  982]


In [14]:
spiking_loader.data_out.connect(linear1.s_in)
leaky3.s_out.connect(out_sink.spikes_in)
out_sink.label_in.connect_from(spiking_loader.label_out)

In [15]:
from sklearn.metrics import confusion_matrix

for i in tqdm(range(num_samples)):
      out_sink.run(condition=RunSteps(num_steps=time_steps),
                  run_cfg=Loihi1SimCfg(select_sub_proc_model=True,
                  select_tag='fixed_pt'))
      leaky1.v.set(np.zeros(leaky1.v.shape))
      leaky1.u.set(np.zeros(leaky1.u.shape))
      leaky2.v.set(np.zeros(leaky2.v.shape))
      leaky2.u.set(np.zeros(leaky2.u.shape))
      leaky3.v.set(np.zeros(leaky3.v.shape))
      leaky3.u.set(np.zeros(leaky3.u.shape))
      ahpc.v.set(np.zeros(ahpc.v.shape))
      ahpc.u.set(np.zeros(ahpc.u.shape))
      linear1.a_buff.set(np.zeros(linear1.a_buff.shape))
      linear2.a_buff.set(np.zeros(linear2.a_buff.shape))
      linear3.a_buff.set(np.zeros(linear3.a_buff.shape))

      updated_weights = linear3.weights.get()

        
ground_truth = val_set[1][:num_samples]
predictions = out_sink.pred_labels.get().astype(int)
# Stop the execution
out_sink.stop()

pre_total_accuracy = np.sum(ground_truth==pre_predictions)/ground_truth.size * 100
pre_train_accuracy = np.sum(ground_truth[:int(num_samples*train_percentage)]==pre_predictions[:int(num_samples*train_percentage)])/int(num_samples*train_percentage) * 100
pre_val_accuracy = np.sum(ground_truth[int(num_samples*train_percentage):]==pre_predictions[int(num_samples*train_percentage):])/int(num_samples*(1-train_percentage)) * 100

total_accuracy = np.sum(ground_truth==predictions)/ground_truth.size * 100
train_accuracy = np.sum(ground_truth[:int(num_samples*train_percentage)]==predictions[:int(num_samples*train_percentage)])/int(num_samples*train_percentage) * 100
val_accuracy = np.sum(ground_truth[int(num_samples*train_percentage):]==predictions[int(num_samples*train_percentage):])/int(num_samples*(1-train_percentage)) * 100

print(f"\nGround truth: {ground_truth}\n"
      f'pre_predictions: {pre_predictions}\n'
      f"Predictions : {predictions}\n"
      f"Pre Total Accuracy    : {pre_total_accuracy} | Total Accuracy    : {total_accuracy}\n"
      f"Pre Training Accuracy : {pre_train_accuracy} | Training Accuracy : {train_accuracy}\n"
      f"Pre Validation Accuracy: {pre_val_accuracy} | Validation Accuracy: {val_accuracy}")

# Calculate the differences between pre_predictions and predictions
# Calculate the confusion matrix

pre_vs_after = confusion_matrix(pre_predictions, predictions)
pre_confusion= confusion_matrix(ground_truth, pre_predictions)
after_confusion= confusion_matrix(ground_truth, predictions)
confusion_matrix 
print("Confusion Matrix:")
print(f"pre confustion:\n {pre_confusion}\n")
print(f"after confusion:\n {after_confusion}\n")
print(f"pre vs after confusion:\n {pre_vs_after}\n")


100%|██████████| 3000/3000 [05:33<00:00,  8.99it/s]


Ground truth: [2 5 4 ... 2 0 1]
pre_predictions: [2 5 3 ... 2 0 6]
Predictions : [2 5 3 ... 2 0 6]
Pre Total Accuracy    : 93.2 | Total Accuracy    : 93.2
Pre Training Accuracy : 92.66666666666666 | Training Accuracy : 92.66666666666666
Pre Validation Accuracy: 94.0 | Validation Accuracy: 94.0
Confusion Matrix:
pre confustion:
 [[404   6   7   2   1   1   1]
 [  3 402   9   4   3   2  11]
 [  9   7 450   0   1   6   2]
 [  3   2   1 399   7   0   1]
 [  3  10   9  35 384   1  11]
 [  0   8   0   2   0 402   1]
 [  1  10   1   5  15   3 355]]

after confusion:
 [[404   6   7   2   1   1   1]
 [  3 402   9   4   3   2  11]
 [  9   7 450   0   1   6   2]
 [  3   2   1 399   7   0   1]
 [  3  10   9  35 384   1  11]
 [  0   8   0   2   0 402   1]
 [  1  10   1   5  15   3 355]]

pre vs after confusion:
 [[423   0   0   0   0   0   0]
 [  0 445   0   0   0   0   0]
 [  0   0 477   0   0   0   0]
 [  0   0   0 447   0   0   0]
 [  0   0   0   0 411   0   0]
 [  0   0   0   0   0 415   0]
 [

In [16]:
print(f"wheights before: {updated_weights}")

wheights before: [[  4.  18.   2. ...   0.  -8.   6.]
 [ -2.  -6.  -2. ...   0.   6.  10.]
 [  4.  -2.  -8. ...   6.  22.  -4.]
 ...
 [ -4.  -4.   2. ...   0.  -2.   6.]
 [-16.   8. -16. ...   8.  -4. -14.]
 [ -2.   6.   2. ...   4.  -2. -12.]]


In [19]:
# Compare two matrices and find the indices of different values
def compare_matrices(matrix1, matrix2):
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same shape to compare.")
    
    diff_indices = np.argwhere(matrix1 != matrix2)
    return diff_indices

# Example usage
matrix1 = linear3.__getattribute__('weights').init
matrix2 = updated_weights

diff_indices = compare_matrices(matrix1, matrix2)
print(f"Indices of different values: {diff_indices}")

Indices of different values: [[  0   1]
 [  0   2]
 [  0   8]
 ...
 [  6 393]
 [  6 395]
 [  6 397]]


In [18]:
np.copyto(bla., original_param['linear3']['weights'])

SyntaxError: invalid syntax (2452119174.py, line 1)

In [26]:
linear3.

RuntimeError: Runtime has not started

In [24]:
np.savez('salva_prova.npz', linear3=linear3., original=matrix2)

AssertionError: '__getstate__' is not a member of 'OutPort' collection of process 'leaky3::LIF'